### AITW-viscosity

AiiDA workflow for automated calculation of shear viscosity of a molecular liquid

#### Links

- <a href="https://github.com/AI-TranspWood/Molecular-liquid-shear-viscosity" target="_blank" rel="noopener noreferrer">Source code</a>
- <a href="https://www.eessi.io/docs/available_software/detail/AITW-viscosity" target="_blank" rel="noopener noreferrer">EESSI</a>

In [ ]:
from utilities import initialize_with_module

await initialize_with_module('AITW-viscosity/1.0.0-foss-2023a', '2023.06')

In [ ]:
# Create/load profile/computer/codes
import os
import ipywidgets as wdg

import utilities_aiida as ua
from common import VISCOSITY_CODES

aiida_setup_output = wdg.Output()

with aiida_setup_output:
    print("Ensure required computer/codes are set up...")

    profile = ua.setup_profile()
    aiida_path = os.environ.get('AIIDA_PATH', None)
    computer = ua.setup_computer()
    codes = ua.setup_codes(code_infos=VISCOSITY_CODES)

    print("Using profile:")
    print(f" - {profile}")
    if aiida_path:
        print(f" - AIIDA_PATH: {aiida_path}")
    print("Using computer:")
    print(f" - {computer}")
    print("Using codes: ")
    for code in codes:
        print(f" - {code}")

    # create_codes_button.on_click(on_create_codes_button_clicked)
    # display(create_codes_button, output_area)

In [ ]:
from aitw_aiida_viscosity.cli.workflows.viscosity import launch_workflow

from common import VISCOSITY_WORKCHAIN_DEFAULTS, VISCOSITY_CODE_MAP
import utilities
import utilities_aiida as ua

with ua.with_aiida_profile():
    widgets_launch, launch_event = utilities.click_cli_widget(
        launch_workflow,
        title="AITW - AiiDA Monomer Viscosity Workchain",
        description_text="",
        override_defaults=VISCOSITY_WORKCHAIN_DEFAULTS,
        custom_groupings={
            # None: [],
            'Simulation time': ['num_steps', 'time_step', 'num_steps_min', 'num_steps_eq', 'averaging_start_time'],
            'Codes': ['acpype_code', 'obabel_code', 'veloxchem_code', 'gmx_code', 'gmx_code_local'],
            'Compute Resources': [
                'num_nodes', 'max_wallclock_seconds', 'num_mpiprocs_per_machine',
                'with_mpi', 'clean_workdir',
                'daemon'
            ],
        },
        code_map=VISCOSITY_CODE_MAP,
        hide_params=['daemon'],
    )
display(widgets_launch)

In [ ]:
import pathlib
import ipywidgets as wdg
from IPython.display import Image

from aiida import orm
import matplotlib.pyplot as plt

import utilities
from common import VISCOSITY_OUTDIR
from utilities_aiida import with_aiida_profile
from aitw_aiida_viscosity.cli.data.viscosity_array import plot_viscosity

plt.tight_layout = lambda *args, **kwargs: None

def get_outputs() -> list[tuple[int, str]]:
    """Get the outputs of the completed workchaisn."""
    with with_aiida_profile():
        qb = orm.QueryBuilder()
        qb.append(orm.ArrayData, filters={'attributes': {'has_key': 'array|deformation_velocities'}})
        return [(node.pk, str(node.pk)) for node, in qb.all()]

def read_func(pk_str: str) -> orm.ArrayData:
    """Read the output data for a given workchain pk."""
    with with_aiida_profile():
        return orm.load_node(int(pk_str))

def label_func(pk: int, array: orm.ArrayData) -> str:
    """Generate a label for the output data based on the workchain pk."""
    with with_aiida_profile():
        wc, _, _ = array.base.links.get_incoming(link_label_filter='viscosity_data').first()

        smile_str = wc.inputs.smiles_string.value
        def_vel = wc.inputs.deform_velocities.value
        nmols = wc.inputs.nmols.value
        num_steps = wc.inputs.num_steps.value
        time_step = wc.inputs.time_step.value
        time_ps = num_steps * time_step

        return f"{smile_str} | nmols: {nmols} | Def Vel: {def_vel} | time (ps): {time_ps:.3f}"

def data_display_func(array: orm.ArrayData):
    """Display the output data for a given workchain."""
    with with_aiida_profile():
        result_dir = pathlib.Path(VISCOSITY_OUTDIR)
        result_dir.mkdir(exist_ok=True, parents=True)

        pk = array.pk
        filename = (result_dir / f"viscosity_data_{pk}.png").as_posix()

        wc, _, _ = array.base.links.get_incoming(link_label_filter='viscosity_data').first()
        print(f"Plotting Viscosity from workchain with inputs:")
        for inp_field in wc.inputs:
            inp_node = getattr(wc.inputs, inp_field)
            try:
                value = inp_node.value
            except AttributeError:
                value = str(inp_node)
            print(f" - {inp_field:>30s}: {value}")

        plot_viscosity([str(pk), '-o', filename], standalone_mode=False)
        display(Image(filename=filename))


visualizer_accordion = utilities.output_visualizer_widget(
    title="Permeability Tensor Results",
    event_observer=launch_event,
    outputs_func=get_outputs,
    read_func=read_func,
    selector_label_func=label_func,
    data_display_func=data_display_func
)

display(visualizer_accordion)
display(aiida_setup_output)


In [ ]:
import utilities
utilities.final_spacer()